# Advanced Python Tutorial: Descriptors, Weak References, and Object Lifetime — New Problems with Walkthrough Solutions

**A second, distinct problem set | 14 advanced problems | step-by-step narrative | runnable code | standard library only**

This notebook follows the teaching style of the supplied lecture: we begin with a seemingly reasonable approach, run a small experiment, interpret what happened, improve the design, and only then assemble a solution. Explanations are intentionally split into many Markdown cells rather than compressed into a single problem/solution block.

**Connection to the source lecture.** That lecture introduces descriptor-managed instance values, `WeakKeyDictionary`, `id(instance)` keys, weak-reference callbacks, hashability restrictions, `__slots__`/`__weakref__`, and validation descriptors. Here we use those concepts as a starting point and investigate **different, more advanced extensions**: callback retention, weak-reference hashing, `__set_name__`, inherited slots, weak-value caches, observer methods, finalizers, serialization, and a composite application.

**Environment.** Python 3.11+ (the slotted-dataclass exercise uses `weakref_slot=True`). Run **Kernel → Restart & Run All**, in order. No external packages, networking, unsafe address dereferencing, or CPython-only reference-count assumptions are required. Object-lifetime assertions isolate references inside functions and call `gc.collect()` where a cycle is involved. Output may differ by Python implementation or display formatting; the assertions express the contracts.

**Conventions.** Each problem introduces its own local classes/functions. Printed text describes behavior; `assert` checks it. The `expect_error` helper makes expected failures part of an otherwise clean notebook. We never introspect a dead object by dereferencing its former memory address.

## Setup — an explicit, reusable error checker

We want the kernel to finish normally even when an example is *supposed* to raise an exception. The helper checks the exception type, optionally checks a message fragment, and fails if no error occurs. This is preferable to presenting a traceback as an unexplained solution.

In [101]:
import copy
import gc
import pickle
import weakref
from dataclasses import dataclass
from typing import Any


def expect_error(expected, action, message_part=None):
    try:
        action()
    except expected as exc:
        if message_part is not None:
            assert message_part in str(exc), str(exc)
        print(f"Expected {type(exc).__name__}: {exc}")
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {expected.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {expected.__name__}, but nothing was raised")


print('Standard-library imports and error helper are ready.')

Standard-library imports and error helper are ready.


---
    ## Problem 01 — Deletion is not destruction

    **The challenge.** Explain why deleting a variable sometimes does **not** clear a descriptor’s weak-key entry. Demonstrate how an alias and a self-cycle change the moment of cleanup.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — predict what a weak-key table actually tracks

A dictionary entry in `WeakKeyDictionary` depends on the **lifetime of the object**, not the existence of any particular variable name. If two names refer to the same instance, deleting one name does not destroy the instance.

We use a hashable, weak-referenceable owner with identity-based equality. That removes the separate hashability limitation from this particular experiment.

In [102]:
def alias_experiment():
    class Owner:
        pass

    table = weakref.WeakKeyDictionary()
    first = Owner()
    alias = first
    table[first] = 'attached state'

    assert len(table) == 1
    del first
    assert len(table) == 1, 'The alias still holds a strong reference'
    assert table[alias] == 'attached state'

    del alias
    gc.collect()
    assert len(table) == 0
    return 'Alias removal and owner destruction are different events.'


print(alias_experiment())

Alias removal and owner destruction are different events.


### Step 2 — now introduce a reference cycle

An object that refers to itself is still reachable from its own reference, even after the outside variable is removed. The cyclic garbage collector can reclaim this unreachable cycle. Calling `gc.collect()` makes the demonstration independent of an automatic collection schedule.

Do not assert the *exact* value returned by `gc.collect()`; that count can include other unreachable objects.

In [103]:
def cycle_experiment():
    class Owner:
        def __init__(self):
            self.loop = self

    table = weakref.WeakKeyDictionary()
    owner = Owner()
    table[owner] = 23
    handle = weakref.ref(owner)
    del owner

    gc.collect()
    assert handle() is None
    assert len(table) == 0
    return 'The unreachable cycle and its weak-key entry were reclaimed.'


print(cycle_experiment())

The unreachable cycle and its weak-key entry were reclaimed.


### Step 3 — solution and design consequence

**Solution:** Weak storage avoids making an owner live *solely because it is a key*. It does not delete your other strong references and does not guarantee immediate reclamation of cycles. For tests, keep owner creation in a function so that notebook output history, exception tracebacks, or accidental aliases cannot keep it alive.

**Challenge extension:** Why might `len(table) == 0` fail if the dictionary value itself refers back to the owner? Because the mapping still holds its **values strongly**; that case was covered in the first problem set and remains an important constraint here.

In [104]:
assert alias_experiment().startswith('Alias')
assert cycle_experiment().startswith('The unreachable')
print('Problem 01 — all checks passed.')

Problem 01 — all checks passed.


---
    ## Problem 02 — A weakref callback that mysteriously never runs

    **The challenge.** Build a callback experiment and find out why a registered callback may disappear before the referent is destroyed.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — distinguish two objects

`weakref.ref(owner, callback)` creates a **weak-reference object**. The owner does not strongly retain that weak-reference object for you. The callback is called when the referent dies **if that weak-reference object is still alive**.

Our first experiment *keeps* the returned weak reference in a local variable. We also keep a normal list of event messages, so we can verify the callback without relying on its printout.

In [105]:
def retained_callback_case():
    class Owner:
        pass

    events = []
    owner = Owner()
    handle = weakref.ref(owner, lambda dead_ref: events.append(dead_ref() is None))
    assert handle() is owner
    del owner
    gc.collect()
    assert handle() is None
    assert events == [True]
    return events


print('Retained weak reference callback:', retained_callback_case())

Retained weak reference callback: [True]


### Step 2 — intentionally drop the weak-reference object

What happens when the weakref is not assigned to a name or stored anywhere? In this controlled example, the temporary reference object disappears before the owner, so the callback is not available when the owner dies.

There is no need to invent a callback-retention scheme involving strong references to owners: simply retain the *weakref object* in a suitable data structure.

In [106]:
def discarded_callback_case():
    class Owner:
        pass

    events = []
    owner = Owner()
    handle = weakref.ref(owner, lambda dead_ref: events.append('destroyed'))
    del handle
    gc.collect()
    del owner
    gc.collect()
    assert events == []
    return events


print('Events with discarded weakref:', discarded_callback_case())

Events with discarded weakref: []


### Step 3 — implement a tiny watch function

The watcher returns the weakref so the *caller* controls its lifetime. The callback receives a dead weakref; it cannot recover the destroyed object by calling `dead_ref()`.

Avoid callback bodies that capture the owner. Capturing it could keep the owner alive indirectly through the callback, defeating the purpose.

In [107]:
def watch_destruction(owner, events, label):
    return weakref.ref(owner, lambda dead: events.append((label, dead() is None)))


def test_watch_destruction():
    class Owner:
        pass

    events = []
    owner = Owner()
    handle = watch_destruction(owner, events, 'record-7')
    assert handle() is owner
    del owner
    gc.collect()
    assert events == [('record-7', True)]
    assert handle() is None


test_watch_destruction()
print('Problem 02 — retained callbacks work; discarded callbacks do not.')

Problem 02 — retained callbacks work; discarded callbacks do not.


**Takeaway.** A weak-reference callback is associated with a weakref **object**, not a durable global notification registration. Keep the weakref alive for as long as you need the callback; never keep the referent alive accidentally through callback arguments or closures.

---
    ## Problem 03 — A hashable weakref to a now-dead object?

    **The challenge.** Investigate weak-reference hashing and explain why a weakref is not a magic hashable wrapper around an unhashable object.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — creating a weakref does not imply it can be hashed

In the lecture, a custom `__eq__` can make instances unhashable. A plain `weakref.ref()` can still point to such an instance. However, computing `hash(ref)` delegates to the referent’s hash behavior and raises if that referent is unhashable.

Test creation and hashing separately; `WeakKeyDictionary` needs **both** weak-referenceability and hashability.

In [108]:
class UnhashableDocument:
    def __eq__(self, other):
        return self is other


doc = UnhashableDocument()
weak_doc = weakref.ref(doc)
assert weak_doc() is doc
expect_error(TypeError, lambda: hash(doc), 'unhashable')
expect_error(TypeError, lambda: hash(weak_doc), 'unhashable')
del weak_doc, doc

Expected TypeError: unhashable type: 'UnhashableDocument'
Expected TypeError: unhashable type: 'UnhashableDocument'


### Step 2 — a surprising before/after difference

For a hashable referent, the weakref caches its hash **the first time you hash the weakref**. If that happens while the owner is alive, the cached hash remains usable after the owner dies. If its first hash is requested only *after* death, a `TypeError` is raised.

This is documented weakref behavior, not permission to use `id()` as a permanent lifetime-unique key.

In [109]:
def cached_hash_case():
    class Owner:
        pass

    obj = Owner()
    handle = weakref.ref(obj)
    recorded_hash = hash(handle)
    del obj
    gc.collect()
    assert handle() is None
    assert hash(handle) == recorded_hash
    return recorded_hash


def first_hash_after_death_case():
    class Owner:
        pass

    obj = Owner()
    handle = weakref.ref(obj)
    del obj
    gc.collect()
    assert handle() is None
    expect_error(TypeError, lambda: hash(handle))


print('Previously cached hash remains available:', isinstance(cached_hash_case(), int))
first_hash_after_death_case()

Previously cached hash remains available: True
Expected TypeError: weak object has gone away


### Step 3 — solution: treat hash and identity as separate concerns

A weakref is a pointer-like observation of lifetime. It is **not** an identity-key dictionary abstraction by itself: referent equality and hashing can still matter while objects are alive. When the owner is unhashable, use instance-owned storage if available or a carefully engineered identity-and-callback store, as explored in the source lecture and earlier exercise set.

**Check:** a weakref to an unhashable object is allowed; a hash operation on that weakref is not.

In [110]:
def weakref_unhashable_check():
    class Owner:
        __hash__ = None

    owner = Owner()
    ref = weakref.ref(owner)
    assert ref() is owner
    expect_error(TypeError, lambda: {ref})


weakref_unhashable_check()
print('Problem 03 — all checks passed.')

Expected TypeError: unhashable type: 'Owner'
Problem 03 — all checks passed.


---
    ## Problem 04 — Teach a descriptor its own attribute name

    **The challenge.** Replace hard-coded storage names with `__set_name__`, explain descriptor reuse across owner classes, and handle dynamic class mutation explicitly.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — the name problem

The source lecture ends by noting that instance-owned storage needs a reliable name. A descriptor object does not automatically know whether it has been installed as `price`, `quantity`, or something else. Since Python 3.6, `__set_name__(owner, name)` is called during class construction for descriptors already present in the class body.

For this exercise we deliberately use the owner’s `__dict__` (not external weak storage), to focus on the attribute naming problem. This **will not** work on instances without a dictionary.

In [111]:
class NamedPositiveInteger:
    def __init__(self):
        self.public_name = None
        self.private_name = None
        self.defining_owner = None

    def __set_name__(self, owner, name):
        if self.defining_owner is not None and owner is not self.defining_owner:
            raise ValueError('Create a fresh descriptor for each unrelated owner class')
        self.defining_owner = owner
        self.public_name = name
        self.private_name = f'_validated_{name}'

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.private_name not in instance.__dict__:
            raise AttributeError(f'{self.public_name} has not been assigned')
        return instance.__dict__[self.private_name]

    def __set__(self, instance, value):
        if type(value) is not int or value <= 0:
            raise ValueError(f'{self.public_name} must be a positive integer')
        instance.__dict__[self.private_name] = value

### Step 2 — use two **different descriptor objects** in one class

Each descriptor receives a different name. Class access should return the descriptor itself; instance access should return the value. Validation should leave a prior valid value unchanged when assignment fails.

In [112]:
class Package:
    width = NamedPositiveInteger()
    height = NamedPositiveInteger()


package = Package()
package.width = 12
package.height = 8
assert (package.width, package.height) == (12, 8)
assert Package.width.public_name == 'width'
assert Package.height.private_name == '_validated_height'
assert Package.width is Package.__dict__['width']
expect_error(ValueError, lambda: setattr(package, 'width', 0), 'width')
assert package.width == 12
print('Descriptor names:', Package.width.public_name, Package.height.public_name)

Expected ValueError: width must be a positive integer
Descriptor names: width height


### Step 3 — inheritance is not the same as descriptor reuse

An inherited descriptor normally continues to use the name and storage associated with the class that originally defined it. The class creation process does not run `__set_name__` a second time merely because a subclass inherits the attribute.

By contrast, installing **the exact same descriptor object** in an unrelated class can silently overwrite its naming configuration. The implementation above rejects that configuration explicitly.

In [113]:
class FragilePackage(Package):
    pass


child = FragilePackage()
child.width = 2
assert child.width == 2
assert FragilePackage.width is Package.width
assert Package.width.defining_owner is Package

other_class = type('OtherClass', (), {})
expect_error(ValueError, lambda: Package.width.__set_name__(other_class, 'length'), 'fresh')

Expected ValueError: Create a fresh descriptor for each unrelated owner class


### Step 4 — dynamic assignments are different

`Class.attribute = descriptor` occurs **after** class creation. Python does not automatically call `__set_name__` in this case. Set up the descriptor and call `__set_name__` deliberately before using it, or put the descriptor in the class body from the beginning.

In [114]:
class Cart:
    pass


late_descriptor = NamedPositiveInteger()
Cart.units = late_descriptor
assert late_descriptor.private_name is None
late_descriptor.__set_name__(Cart, 'units')
cart = Cart()
cart.units = 9
assert cart.units == 9
print('Problem 04 — naming, inheritance, and late assignment checks passed.')

Problem 04 — naming, inheritance, and late assignment checks passed.


**Completed solution.** `__set_name__` solves attribute naming for a descriptor installed at class creation. It does **not** solve memory retention or slotted storage by itself. We will deliberately switch storage strategies as later problems demand.

---
    ## Problem 05 — Class-level access and subclass-aware metadata

    **The challenge.** Build a reusable read-only descriptor that reports information about the accessing class without storing owner instances.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — what does `__get__` receive?

Calling `Base.descriptor` invokes `__get__(None, Base)` for ordinary descriptor access. Calling `Sub.descriptor` likewise passes the subclass as `owner`. Calling `instance.descriptor` passes the actual instance and its class as the owner.

The descriptor should never confuse a class with an instance, and it should avoid storing the subclass in an accumulating strong-reference registry.

In [115]:
class ClassLineage:
    def __set_name__(self, owner, name):
        self.attribute_name = name
        self.declared_in = owner

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return tuple(cls.__name__ for cls in type(instance).__mro__ if cls is not object)


class BaseNode:
    lineage = ClassLineage()


class SpecializedNode(BaseNode):
    pass


assert BaseNode.lineage is SpecializedNode.lineage
assert BaseNode.lineage.attribute_name == 'lineage'
assert BaseNode().lineage == ('BaseNode',)
assert SpecializedNode().lineage == ('SpecializedNode', 'BaseNode')
print('Subclass lineage:', SpecializedNode().lineage)

Subclass lineage: ('SpecializedNode', 'BaseNode')


### Step 2 — prevent accidental instance overrides

The current descriptor defines only `__get__`: it is a **non-data descriptor**. For a normal object with `__dict__`, a value stored under the same instance key can shadow the descriptor.

To make lineage read-only and take precedence over the instance dictionary, add `__set__` that rejects assignment. This changes it into a **data descriptor**. Here, we build a separate final version so the before/after behavior is visible.

In [116]:
node = BaseNode()
node.__dict__['lineage'] = ('spoofed',)
assert node.lineage == ('spoofed',)


class ReadOnlyClassLineage(ClassLineage):
    def __set__(self, instance, value):
        raise AttributeError('lineage is computed and read-only')


class ProtectedNode:
    lineage = ReadOnlyClassLineage()


protected = ProtectedNode()
protected.__dict__['lineage'] = ('spoofed',)
assert protected.lineage == ('ProtectedNode',)
expect_error(AttributeError, lambda: setattr(protected, 'lineage', ('fake',)), 'read-only')
print('Data-descriptor precedence preserves the computed lineage.')

Expected AttributeError: lineage is computed and read-only
Data-descriptor precedence preserves the computed lineage.


### Step 3 — conclusion

The `owner` argument identifies the class involved in descriptor lookup; `instance is None` identifies class-level access. Make a descriptor a data descriptor when its computed value must not be shadowed by an instance dictionary. The weak-reference issues in the source concern **stored per-instance state**; a pure computed descriptor like this needs no weak table at all.

In [117]:
assert ProtectedNode.lineage is ProtectedNode.__dict__['lineage']
assert not hasattr(ProtectedNode.lineage, '__delete__')
assert hasattr(ProtectedNode.lineage, '__set__')
print('Problem 05 — all checks passed.')

Problem 05 — all checks passed.


---
    ## Problem 06 — Slots and inheritance: three apparently similar classes

    **The challenge.** Determine exactly which instances have a dictionary and which can be weakly referenced when `__slots__` are combined with inheritance.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — putting `__slots__` on a child does not retroactively remove the parent’s dictionary

An ordinary base class normally provides its instances with `__dict__` and weak-reference support. A subclass that declares `__slots__` retains those **inherited** facilities. This often surprises people who interpret `__slots__` as a global ban on `__dict__`.

In [118]:
class OrdinaryBase:
    pass


class SlottedChild(OrdinaryBase):
    __slots__ = ('code',)


one = SlottedChild()
one.code = 'A'
one.extra = 42
assert hasattr(one, '__dict__')
assert one.__dict__['extra'] == 42
assert weakref.ref(one)() is one
print('Slotted child retains its ordinary base-class instance dictionary.')

Slotted child retains its ordinary base-class instance dictionary.


### Step 2 — a genuinely slotted base changes the outcome

A base with `__slots__ = ()` supplies neither `__dict__` nor `__weakref__`. A subclass that adds `__weakref__` can be weak-referenced but, if no class supplies `__dict__`, cannot store arbitrary new attributes.

The slot for weak references needs to be provided only **once** in an inheritance chain.

In [119]:
class TinyBase:
    __slots__ = ()


class TinyChild(TinyBase):
    __slots__ = ('code', '__weakref__')


small = TinyChild()
small.code = 'B'
assert weakref.ref(small)() is small
assert not hasattr(small, '__dict__')
expect_error(AttributeError, lambda: setattr(small, 'other', 99))


class TinyGrandchild(TinyChild):
    __slots__ = ('revision',)


grandchild = TinyGrandchild()
grandchild.code = 'C'
grandchild.revision = 3
assert weakref.ref(grandchild)() is grandchild
assert not hasattr(grandchild, '__dict__')

Expected AttributeError: 'TinyChild' object has no attribute 'other' and no __dict__ for setting new attributes


### Step 3 — the empty-slots trap for subclasses

A subclass that **omits `__slots__` altogether** usually receives an instance dictionary even if its base was fully slotted. That might be useful, but it invalidates an assumption that the entire hierarchy has a fixed-memory, dictionary-free layout.

In [120]:
class FlexibleGrandchild(TinyChild):
    pass


flexible = FlexibleGrandchild()
flexible.unplanned_field = 'allowed'
assert flexible.__dict__ == {'unplanned_field': 'allowed'}
assert weakref.ref(flexible)() is flexible
print('No __slots__ declaration on a subclass permits an instance dictionary.')

No __slots__ declaration on a subclass permits an instance dictionary.


### Step 4 — modern dataclasses provide an explicit option

On Python 3.11+, `@dataclass(slots=True, weakref_slot=True, eq=False)` builds a slotted, weak-referenceable class. We request `eq=False` because default generated equality generally makes mutable dataclass instances unhashable, which matters for `WeakKeyDictionary`.

The dataclass feature below is an **extension beyond the supplied lecture**, which uses explicit `__slots__` declarations.

In [121]:
@dataclass(slots=True, weakref_slot=True, eq=False)
class Sensor:
    label: str
    reading: float


sensor = Sensor('north', 22.5)
assert weakref.ref(sensor)() is sensor
assert not hasattr(sensor, '__dict__')
assert hash(sensor) == object.__hash__(sensor)
measurements = weakref.WeakKeyDictionary()
measurements[sensor] = 'calibrated'
assert measurements[sensor] == 'calibrated'
print('Problem 06 — inheritance and slotted-dataclass checks passed.')

Problem 06 — inheritance and slotted-dataclass checks passed.


**Decision rule:** When you control an owner class, inspect its **whole inheritance hierarchy**, not merely the immediate `__slots__` line. External weak storage requires weak-referenceable owners; `WeakKeyDictionary` adds a hashability requirement.

---
    ## Problem 07 — Use a weak proxy, then watch it fail safely

    **The challenge.** Create a non-owning proxy for an object, distinguish it from a normal weakref, and handle the inevitable `ReferenceError`.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — when `weakref.proxy` is useful

A `weakref.ref` must be called to retrieve the object. A `weakref.proxy` instead delegates ordinary operations (attribute access, method calls) while the original object exists. It does **not** keep the referent alive.

Use this when collaborators should be able to interact with a still-live object but must not own its lifetime.

In [122]:
class Job:
    def __init__(self, name):
        self.name = name
        self.runs = 0

    def run(self):
        self.runs += 1
        return self.runs


job = Job('sync')
job_proxy = weakref.proxy(job)
assert job_proxy.name == 'sync'
assert job_proxy.run() == 1
assert job.runs == 1
print('Proxy forwards operations while its target lives.')

Proxy forwards operations while its target lives.


### Step 2 — the proxy is not a durable owner

Delete the last strong reference to the job. The proxy remains a Python object, but access through it now raises `ReferenceError`. We deliberately do not use an expression that prints the proxy itself after death: even representation can require the vanished referent.

In [123]:
del job
gc.collect()
expect_error(ReferenceError, lambda: job_proxy.name)
expect_error(ReferenceError, lambda: job_proxy.run())

Expected ReferenceError: weakly-referenced object no longer exists
Expected ReferenceError: weakly-referenced object no longer exists


### Step 3 — solution: use an explicit availability boundary

A normal weakref is often easier when clients must **test whether the object is alive** before using it. Retrieve the referent into a temporary local variable, then work with it; doing so keeps the object alive during that operation.

Avoid a separate `if handle() is not None` and later `handle().method()` in concurrent code: the two dereferences need not return the same result.

In [124]:
def safe_run(handle):
    current = handle()  # A temporary strong reference for this operation.
    if current is None:
        return None
    return current.run()


def test_safe_run():
    owner = Job('backup')
    handle = weakref.ref(owner)
    assert safe_run(handle) == 1
    del owner
    gc.collect()
    assert safe_run(handle) is None


test_safe_run()
print('Problem 07 — proxy failure and safe weakref use demonstrated.')

Problem 07 — proxy failure and safe weakref use demonstrated.


---
    ## Problem 08 — A weak-value cache that seems to forget everything

    **The challenge.** Build a cache with `WeakValueDictionary`, explain the difference from weak keys, and choose an API that does not promise persistent storage.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — invert the lecture’s weak-key idea

`WeakKeyDictionary` holds keys weakly and values strongly. `WeakValueDictionary` holds keys strongly and values weakly. The latter can be useful for memoizing large objects **only while some other part of the program still owns those objects**.

A weak-value cache cannot guarantee that a retrieved object will remain cached after all external references disappear.

In [125]:
class ParsedSchema:
    __slots__ = ('name', 'fields', '__weakref__')

    def __init__(self, name, fields):
        self.name = name
        self.fields = tuple(fields)


cache = weakref.WeakValueDictionary()
cache['orders'] = ParsedSchema('orders', ['id', 'amount'])
gc.collect()
assert 'orders' not in cache
print('A weak-value cache entry can vanish immediately without an external owner.')

A weak-value cache entry can vanish immediately without an external owner.


### Step 2 — give the cached value an outside owner

This time we keep a normal `schema` variable, insert it into the cache, and check identity. We can also create a normal weakref to observe exactly when the schema becomes collectible.

In [126]:
def pinned_cache_case():
    local_cache = weakref.WeakValueDictionary()
    schema = ParsedSchema('orders', ['id', 'amount'])
    witness = weakref.ref(schema)
    local_cache[schema.name] = schema
    assert local_cache['orders'] is schema

    del schema
    gc.collect()
    assert witness() is None
    assert 'orders' not in local_cache
    return len(local_cache)


assert pinned_cache_case() == 0

### Step 3 — implement get-or-create without claiming persistent memoization

A caller should hold the result as long as it needs the object. The cache is merely an opportunity to reuse a live instance. There is no requirement that separate calls separated by loss of all strong references return the same object.

In [127]:
class SchemaPool:
    def __init__(self):
        self._schemas = weakref.WeakValueDictionary()
        self.build_count = 0

    def get(self, name, field_names):
        signature = (name, tuple(field_names))  # A stable, immutable cache key.
        schema = self._schemas.get(signature)
        if schema is None:
            self.build_count += 1
            schema = ParsedSchema(name, signature[1])
            self._schemas[signature] = schema
        return schema


pool = SchemaPool()
first = pool.get('users', ['id', 'email'])
second = pool.get('users', ['id', 'email'])
assert first is second
assert pool.build_count == 1
assert pool.get('users', ['id']) is not first
assert pool.build_count == 2
print('Pool builds:', pool.build_count)

Pool builds: 2


### Step 4 — cleanup contract

The local `first` and `second` variables keep one schema alive. Deleting both allows the weak-value mapping to shed that entry. A different schema may still remain because a prior notebook expression or variable references it; for predictable assertions, isolate the complete scenario in a function.

In [128]:
def pool_cleanup_case():
    local_pool = SchemaPool()
    saved = local_pool.get('metrics', ['p50', 'p95'])
    witness = weakref.ref(saved)
    assert len(local_pool._schemas) == 1
    del saved
    gc.collect()
    assert witness() is None
    assert len(local_pool._schemas) == 0
    replacement = local_pool.get('metrics', ['p50', 'p95'])
    assert replacement.fields == ('p50', 'p95')
    assert local_pool.build_count == 2


pool_cleanup_case()
print('Problem 08 — all checks passed.')

Problem 08 — all checks passed.


**Important:** Values must support weak references. A plain `list` cannot be directly cached as a weak value; wrap its contents in a weak-referenceable object such as `ParsedSchema`. This is one reason a cache's value type is part of its interface.

In [129]:
expect_error(TypeError, lambda: weakref.ref([]), 'weak reference')
expect_error(TypeError, lambda: weakref.WeakValueDictionary({'x': []}), 'weak reference')

Expected TypeError: cannot create weak reference to 'list' object
Expected TypeError: cannot create weak reference to 'list' object


---
    ## Problem 09 — Track the living population using WeakSet

    **The challenge.** Maintain a live-object registry without retaining instances, then diagnose why a logically equal or unhashable object may behave unexpectedly.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — use identity semantics deliberately

A `WeakSet` is a convenient collection of objects that supports membership and iteration without owning their lifetimes. The elements must be hashable and weak-referenceable. Use identity equality if your goal is "which exact instances are alive?"

In [130]:
class LiveConnection:
    __slots__ = ('name', '__weakref__')

    def __init__(self, name):
        self.name = name


def connection_registry_case():
    live = weakref.WeakSet()
    a = LiveConnection('primary')
    b = LiveConnection('replica')
    live.add(a)
    live.add(b)
    assert len(live) == 2
    assert a in live and b in live
    witness = weakref.ref(a)
    del a
    gc.collect()
    assert witness() is None
    assert len(live) == 1
    assert b in live


connection_registry_case()
print('A WeakSet tracks membership without extending lifetimes.')

A WeakSet tracks membership without extending lifetimes.


### Step 2 — two independent prerequisites

The lecture demonstrated that `__eq__` without an appropriate `__hash__` makes objects unhashable. `WeakSet` has the same restriction as `WeakKeyDictionary`; being weak-referenceable is not enough.

In [131]:
class EqualConnection:
    __slots__ = ('name', '__weakref__')

    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return isinstance(other, EqualConnection) and self.name == other.name


unhashable = EqualConnection('same')
assert weakref.ref(unhashable)() is unhashable
expect_error(TypeError, lambda: weakref.WeakSet([unhashable]), 'unhashable')
del unhashable

Expected TypeError: unhashable type: 'EqualConnection'


### Step 3 — equality-aware sets can collapse distinct owners

A custom hash consistent with equality makes the class acceptable to `WeakSet`, but two distinct objects that compare equal can occupy only one logical set slot. If you are building a registry of **instances**, a value-based equality definition may be the wrong key semantics. This is an illustration of standard set equality rather than a recommendation to force `__hash__ = object.__hash__` onto value-based equality.

In [132]:
class ValueConnection:
    __slots__ = ('name', '__weakref__')

    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return isinstance(other, ValueConnection) and self.name == other.name

    def __hash__(self):
        return hash(self.name)


one_connection = ValueConnection('same')
another_connection = ValueConnection('same')
assert one_connection is not another_connection
members = weakref.WeakSet([one_connection, another_connection])
assert len(members) == 1
print('Two equal but different objects occupy one logical set entry.')

Two equal but different objects occupy one logical set entry.


### Step 4 — solution

Use `WeakSet` when ordinary set semantics are what you need and live membership is enough. For a registry requiring strict identity for unhashable owners, the original lecture’s `id()`-plus-weak-callback approach (with proper cleanup and stale-callback protection) addresses a different contract. Do not silently change a value object’s `__hash__` in order to fit it into a registry.

In [133]:
assert one_connection in members
assert another_connection in members
# Keep both strong references: temporary WeakSet elements may die immediately.
left, right = LiveConnection('same'), LiveConnection('same')
identity_set = weakref.WeakSet([left, right])
assert len(identity_set) == 2
print('Problem 09 — all checks passed.')

Problem 09 — all checks passed.


---
    ## Problem 10 — Event subscribers: a bound method is a lifetime trap

    **The challenge.** Construct a weak-observer event bus using `WeakMethod`, while preserving ordinary event delivery and pruning dead subscribers.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — storing a bound method strongly also stores its instance strongly

`subscriber.on_event` is a bound-method object; it contains a strong reference to `subscriber`. A normal list of bound methods can therefore prevent subscribers from being collected even after other references are removed.

A plain `weakref.ref(bound_method)` is not a reliable fix: bound-method objects are commonly short-lived temporaries. `weakref.WeakMethod` is specialized for weakly referring to bound methods by tracking their function and instance.

In [134]:
class Subscriber:
    __slots__ = ('name', 'events', '__weakref__')

    def __init__(self, name):
        self.name = name
        self.events = []

    def on_event(self, payload):
        self.events.append(payload)


def strong_method_trap():
    subscriptions = []
    sub = Subscriber('alice')
    witness = weakref.ref(sub)
    subscriptions.append(sub.on_event)
    del sub
    gc.collect()
    assert witness() is not None, 'The bound method owns its instance'
    subscriptions.clear()
    gc.collect()
    assert witness() is None


strong_method_trap()
print('A bound method in a list kept the subscriber alive.')

A bound method in a list kept the subscriber alive.


### Step 2 — create a weak method, not a weak temporary method object

Call a `WeakMethod` to obtain a usable bound method **only while the instance lives**. After the instance dies, calling the weak method returns `None`. The temporary bound method you retrieve is a strong reference during the call, which is exactly what we need to prevent disappearance halfway through event delivery.

In [135]:
def weak_method_case():
    sub = Subscriber('bob')
    method_ref = weakref.WeakMethod(sub.on_event)
    method = method_ref()
    assert method is not None
    method('hello')
    assert sub.events == ['hello']

    del method  # Otherwise this bound method would keep sub alive.
    del sub
    gc.collect()
    assert method_ref() is None


weak_method_case()

### Step 3 — full solution: a small event bus

The bus accepts bound methods of ordinary Python instances. It stores `WeakMethod` objects, never bound methods. On publication it obtains a temporary strong callable, invokes it, then keeps only live weak references.

This deliberately does **not** support arbitrary functions or callable instances; a production API would define separate ownership policies for each listener type. Keeping the contract narrow makes the lifetime guarantee explicit.

In [136]:
class WeakEventBus:
    def __init__(self):
        self._listeners = []

    def subscribe(self, bound_method):
        if not (hasattr(bound_method, '__self__') and bound_method.__self__ is not None):
            raise TypeError('subscribe expects a bound instance method')
        self._listeners.append(weakref.WeakMethod(bound_method))

    def publish(self, message):
        survivors = []
        for method_ref in tuple(self._listeners):
            method = method_ref()
            if method is None:
                continue
            survivors.append(method_ref)
            method(message)
            del method
        self._listeners = survivors

    @property
    def subscription_count(self):
        return len(self._listeners)


def test_weak_event_bus():
    bus = WeakEventBus()
    a = Subscriber('a')
    b = Subscriber('b')
    handle_a = weakref.ref(a)
    bus.subscribe(a.on_event)
    bus.subscribe(b.on_event)
    assert bus.subscription_count == 2
    bus.publish('first')
    assert a.events == ['first'] and b.events == ['first']

    del a
    gc.collect()
    assert handle_a() is None
    bus.publish('second')
    assert b.events == ['first', 'second']
    assert bus.subscription_count == 1
    expect_error(TypeError, lambda: bus.subscribe(lambda message: None), 'bound')


test_weak_event_bus()
print('Problem 10 — observer lifetime and delivery checks passed.')

Expected TypeError: subscribe expects a bound instance method
Problem 10 — observer lifetime and delivery checks passed.


### Step 4 — examine an important scope assumption

This bus assumes callbacks will not mutate the subscription list in ways that need to be immediately reflected in the same publication, and it is not designed for multithreaded dispatch. Those are **separate API questions**. The central lesson is that the lifetime of a **method object** is different from the lifetime of the **subscriber** it binds.

---
    ## Problem 11 — Finalize a resource without accidentally preserving its owner

    **The challenge.** Use `weakref.finalize` to close an external resource when an owner becomes unreachable, and implement an explicit close path that runs only once.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — a finalizer needs enough information to clean up without holding the owner

`weakref.finalize(owner, func, *args)` arranges for `func(*args)` to run when the owner is collected (or when the finalizer is called explicitly). It keeps the cleanup callable and arguments alive. Therefore the callable or arguments **must not hold a strong reference to owner**.

A separate resource object is fine if that resource does not point back to its owner. The finalizer can strongly hold `resource.close`, because keeping the *resource* alive is necessary to close it.

In [137]:
class Resource:
    def __init__(self):
        self.closed = False
        self.close_count = 0

    def close(self):
        if not self.closed:
            self.closed = True
            self.close_count += 1


class ResourceOwner:
    __slots__ = ('resource', '_finalizer', '__weakref__')

    def __init__(self, resource):
        self.resource = resource
        # resource.close retains resource, NOT ResourceOwner.
        self._finalizer = weakref.finalize(self, resource.close)

    def close(self):
        self._finalizer()  # Calling a finalize object at most once runs the callback.

    @property
    def is_active(self):
        return self._finalizer.alive

### Step 2 — manual cleanup should not run twice

A cleanup API is easier to reason about if a user can close a resource explicitly. Calling the finalizer runs its callback and marks the finalizer no longer alive; later garbage collection does not repeat the cleanup.

In [138]:
manual_resource = Resource()
manual_owner = ResourceOwner(manual_resource)
assert manual_owner.is_active
manual_owner.close()
manual_owner.close()
assert manual_resource.closed
assert manual_resource.close_count == 1
assert not manual_owner.is_active
del manual_owner
gc.collect()
assert manual_resource.close_count == 1
print('Explicit close is idempotent.')

Explicit close is idempotent.


### Step 3 — test automatic cleanup with separate owner and resource variables

We retain `resource` only to inspect the result after owner destruction. Because the resource does not refer back to its owner, this does not keep the owner alive. In contrast, `weakref.finalize(self, self.close)` would strongly retain the bound method and therefore `self`, preventing ordinary collection.

In [139]:
def automatic_finalizer_case():
    resource = Resource()
    owner = ResourceOwner(resource)
    witness = weakref.ref(owner)
    assert not resource.closed

    del owner
    gc.collect()
    assert witness() is None
    assert resource.closed
    assert resource.close_count == 1


automatic_finalizer_case()
print('Finalizer ran when the owner became unreachable.')

Finalizer ran when the owner became unreachable.


### Step 4 — solution limitations

Prefer explicit `with` blocks or `close()` methods for time-sensitive resource release. Garbage collection timing can vary, and interpreter shutdown is a separate scenario. `weakref.finalize` is a safety net rather than a substitute for explicit lifecycle management.

For cleanup callbacks, avoid printing tracebacks or throwing exceptions: exceptions from weak-reference cleanup may be reported as unraisable instead of propagating to your `del` statement.

In [140]:
def finalizer_once_case():
    resource = Resource()
    owner = ResourceOwner(resource)
    owner.close()
    assert resource.close_count == 1
    del owner
    gc.collect()
    assert resource.close_count == 1


finalizer_once_case()
print('Problem 11 — all checks passed.')

Problem 11 — all checks passed.


---
    ## Problem 12 — A weak-backed descriptor that serializes incorrectly

    **The challenge.** Explain why external descriptor state is not automatically in the instance’s pickle representation and implement correct pickle/deepcopy round trips.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — construct an owner with no instance dictionary

The lecture stores descriptor data outside instances. That is compatible with `__slots__`, but it means a serializer inspecting only instance-owned state cannot discover the descriptor-managed fields by magic. We will use **identity-hashable** owners and `WeakKeyDictionary` so that this problem isolates serialization rather than rebuilding the unhashable identity store from the earlier exercise set.

In [141]:
class WeakScore:
    def __init__(self, minimum=0, maximum=100):
        self.minimum = minimum
        self.maximum = maximum
        self._scores = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._scores[instance]
        except KeyError:
            raise AttributeError('score has not been assigned') from None

    def __set__(self, instance, value):
        if type(value) is not int or not self.minimum <= value <= self.maximum:
            raise ValueError('score must be an integer in the configured interval')
        self._scores[instance] = value


class SerializablePlayer:
    __slots__ = ('__weakref__',)
    score = WeakScore()

    def __init__(self, score):
        self.score = score

    def __getstate__(self):
        # Serialize logical state, not the descriptor's weakref machinery.
        return {'score': self.score}

    def __setstate__(self, state):
        # Rehydrate through validation and register the new owner instance.
        self.score = state['score']

### Step 2 — confirm there really is no `__dict__`

The class has no per-instance slots beyond `__weakref__`; the score resides only in a descriptor-level mapping. The instance is nevertheless weak-referenceable and identity-hashable because we did not override equality.

In [142]:
original_player = SerializablePlayer(91)
assert not hasattr(original_player, '__dict__')
assert original_player.score == 91
assert SerializablePlayer.score._scores[original_player] == 91
assert SerializablePlayer.score is SerializablePlayer.__dict__['score']
print('Logical state:', original_player.__getstate__())

Logical state: {'score': 91}


### Step 3 — pickle and deepcopy must produce *different owners* with the same logical data

Pickling a class-level descriptor itself is not the goal. Rather, `__getstate__` extracts an ordinary dictionary of serializable values, and `__setstate__` routes them back through the descriptor for the new instance. Because the reconstructed instances are different objects, they should have independent descriptor entries.

In [143]:
pickled_player = pickle.loads(pickle.dumps(original_player))
copied_player = copy.deepcopy(original_player)

assert pickled_player is not original_player
assert copied_player is not original_player
assert len({id(original_player), id(pickled_player), id(copied_player)}) == 3
assert (pickled_player.score, copied_player.score) == (91, 91)

pickled_player.score = 84
copied_player.score = 72
assert original_player.score == 91
assert pickled_player.score == 84
assert copied_player.score == 72
print('Independent round-trip scores:', original_player.score, pickled_player.score, copied_player.score)

Independent round-trip scores: 91 84 72


### Step 4 — validation remains enforced during reconstruction

A malformed state dictionary must not bypass the descriptor's validation. This is a useful defensive check for internal transformations, but remember that **pickle data from untrusted sources is unsafe to load**, regardless of your `__setstate__` method. Never use pickle as a safe interchange format for arbitrary user input.

In [144]:
blank = SerializablePlayer.__new__(SerializablePlayer)
expect_error(ValueError, lambda: blank.__setstate__({'score': 1000}), 'interval')
expect_error(AttributeError, lambda: blank.score, 'assigned')
blank.__setstate__({'score': 65})
assert blank.score == 65
print('Problem 12 — serialization and validation checks passed.')

Expected ValueError: score must be an integer in the configured interval
Expected AttributeError: score has not been assigned
Problem 12 — serialization and validation checks passed.


---
    ## Problem 13 — A derived descriptor whose cached value goes stale

    **The challenge.** Build a weak-key-backed computed field with an explicit invalidation contract and verify that each owner has an independent cache.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — introduce the stale-cache problem

Suppose the result of a costly calculation depends on an owner’s `quantity` and `unit_price`. A weak-key mapping can keep a cached **immutable numeric result** without retaining the owner. But weak storage does not automatically know when a dependency changes.

The descriptor will provide `invalidate(owner)` and forbid direct assignment. The owner’s property setters will invalidate the cache after successful updates.

In [145]:
class WeakCachedTotal:
    def __init__(self):
        self._cache = weakref.WeakKeyDictionary()
        self._computations = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if instance not in self._cache:
            # A tuple or integer containing only value data will not retain owner.
            result = instance.quantity * instance.unit_price
            self._cache[instance] = result
            self._computations[instance] = self._computations.get(instance, 0) + 1
        return self._cache[instance]

    def __set__(self, instance, value):
        raise AttributeError('total is computed; change quantity or unit_price')

    def invalidate(self, instance):
        self._cache.pop(instance, None)

    def computation_count(self, instance):
        return self._computations.get(instance, 0)

### Step 2 — implement controlled dependencies

The owner uses slots and supports weak references. It does not define value-based equality, so it keeps identity-based hashing. We invalidate after validation has succeeded; invalid input must not alter the object or its cached result.

In [146]:
class OrderLine:
    __slots__ = ('_quantity', '_unit_price', '__weakref__')
    total = WeakCachedTotal()

    def __init__(self, quantity, unit_price):
        self.quantity = quantity
        self.unit_price = unit_price

    @property
    def quantity(self):
        return self._quantity

    @quantity.setter
    def quantity(self, value):
        if type(value) is not int or value < 0:
            raise ValueError('quantity must be a nonnegative integer')
        self._quantity = value
        type(self).total.invalidate(self)

    @property
    def unit_price(self):
        return self._unit_price

    @unit_price.setter
    def unit_price(self, value):
        if type(value) not in (int, float) or value < 0:
            raise ValueError('unit_price must be nonnegative and numeric')
        self._unit_price = value
        type(self).total.invalidate(self)

### Step 3 — prove cache reuse, invalidation, and atomic rejection

Reading `total` twice should perform one calculation. Changing quantity should force one new calculation. A rejected assignment should leave both the old value and the current cache untouched.

In [147]:
line = OrderLine(3, 9)
assert line.total == 27
assert line.total == 27
assert OrderLine.total.computation_count(line) == 1

line.quantity = 4
assert line.total == 36
assert OrderLine.total.computation_count(line) == 2

expect_error(ValueError, lambda: setattr(line, 'quantity', -5), 'quantity')
assert line.quantity == 4
assert line.total == 36
assert OrderLine.total.computation_count(line) == 2
expect_error(AttributeError, lambda: setattr(line, 'total', 999), 'computed')
print('Computed only when dependencies changed.')

Expected ValueError: quantity must be a nonnegative integer
Expected AttributeError: total is computed; change quantity or unit_price
Computed only when dependencies changed.


### Step 4 — independent owners and automatic cleanup

Cache entries represent per-owner data. Keep the checks in a function: if the notebook stores an owner in its output or a global, `gc.collect()` cannot override that strong reference. The descriptor itself stays on the class and outlives the instances.

In [148]:
def cache_lifetime_case():
    descriptor = OrderLine.total
    start = len(descriptor._cache)
    owner = OrderLine(5, 2)
    peer = OrderLine(7, 3)
    witness = weakref.ref(owner)
    assert owner.total == 10
    assert peer.total == 21
    assert len(descriptor._cache) == start + 2
    del owner
    gc.collect()
    assert witness() is None
    assert len(descriptor._cache) == start + 1
    assert peer.total == 21
    del peer
    gc.collect()
    assert len(descriptor._cache) == start


cache_lifetime_case()
print('Problem 13 — cache correctness and cleanup checks passed.')

Problem 13 — cache correctness and cleanup checks passed.


**Design warning.** A weak-key mapping stores its **values strongly**. A cached result containing a reference to its owner can prevent owner collection. We cache a numeric value here, which does not contain the owner. In a production cache, document the payload’s ownership graph as carefully as its hashability requirements.

---
    ## Problem 14 — Capstone: a live registry with a validated, serializable, derived record

    **The challenge.** Combine independent weak-backed fields, a weak live-object registry, derived-value invalidation, and explicit serialization into one small coherent system.

    **How to use this tutorial:** Read each experiment and predict its outcome before running it. The answer unfolds in small steps, followed by a finished solution and executable checks. This problem is independent of the others unless explicitly stated.

### Step 1 — write down the contract before any code

A telemetry record must support these operations: a validated integer `reading` in `[0, 100]`; a user-controlled `scale` of either `1` or `10`; a derived `scaled_reading`; a registry that observes living records without keeping them alive; and pickle/deepcopy that preserve values but create new owners.

We will intentionally restrict this design to **identity-hashable, weak-referenceable** records. The supplied lecture’s unhashable-owner identity store is an alternative for other constraints; introducing it again would obscure this capstone’s focus on composing clean contracts.

**Invariants:** descriptor storage and registry cannot retain owners; derived caches must be invalidated after updates; reconstituted records are separate instances; invalid changes leave state untouched.

### Step 2 — implement a self-contained descriptor for validated readings

The class-level descriptor stores one value per instance in a `WeakKeyDictionary`. Its `__get__` returns itself on class access and raises `AttributeError` for an unreadable, uninitialized field. `__set__` performs validation **before mutation**.

In [149]:
class BoundedReading:
    def __init__(self, low=0, high=100):
        self.low, self.high = low, high
        self._values = weakref.WeakKeyDictionary()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError(f'{self.name} is not initialized') from None

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError(f'{self.name} must be an integer in [{self.low}, {self.high}]')
        self._values[instance] = value

### Step 3 — implement a weak registry and a derived cache

The registry is an independent helper rather than a global list of records. `WeakSet` is appropriate because our record uses identity hashing. The derived descriptor caches only an integer, not a reference to the record, and its invalidation operation is explicit.

In [150]:
class LiveRegistry:
    def __init__(self):
        self._live = weakref.WeakSet()

    def register(self, record):
        self._live.add(record)

    def count(self):
        return len(self._live)

    def snapshot(self):
        # This list temporarily *does* hold strong references. Use and release it.
        return list(self._live)


class ScaledReading:
    def __init__(self):
        self._values = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            result = instance.reading * instance.scale
            self._values[instance] = result
            return result

    def __set__(self, instance, value):
        raise AttributeError('scaled_reading is derived')

    def invalidate(self, instance):
        self._values.pop(instance, None)

### Step 4 — assemble the slotted record

The class has only the scale slot and `__weakref__`. It has no `__dict__`, and it inherits identity hashability. The raw reading descriptor does not know about the derived cache, so directly exposing it as `.reading` would allow stale values after assignment.

To avoid that bypass, we introduce a **private descriptor field** `_reading_value` and a public property `reading` that routes every change through invalidation. We also validate `scale` before mutation.

In [151]:
class TelemetryRecord:
    __slots__ = ('_scale', '__weakref__')
    _reading_value = BoundedReading(0, 100)
    scaled_reading = ScaledReading()

    def __init__(self, reading, scale=1, registry=None):
        self.reading = reading
        self.scale = scale
        if registry is not None:
            registry.register(self)

    @property
    def reading(self):
        return self._reading_value

    @reading.setter
    def reading(self, value):
        type(self)._reading_value.__set__(self, value)
        type(self).scaled_reading.invalidate(self)

    @property
    def scale(self):
        return self._scale

    @scale.setter
    def scale(self, value):
        if type(value) is not int or value not in (1, 10):
            raise ValueError('scale must be exactly 1 or 10')
        self._scale = value
        type(self).scaled_reading.invalidate(self)

    def __getstate__(self):
        return {'reading': self.reading, 'scale': self.scale}

    def __setstate__(self, state):
        self.reading = state['reading']
        self.scale = state['scale']

### Step 5 — run everyday invariants first

A simple functional check is necessary before testing garbage collection. If state or cache invalidation is incorrect, lifetime assertions will only make debugging harder. Test two different records, invalid assignments, and class-level access.

In [152]:
registry = LiveRegistry()
record = TelemetryRecord(12, scale=10, registry=registry)
peer_record = TelemetryRecord(8, scale=1, registry=registry)
assert not hasattr(record, '__dict__')
assert weakref.ref(record)() is record
assert registry.count() == 2
assert (record.scaled_reading, peer_record.scaled_reading) == (120, 8)
assert TelemetryRecord.scaled_reading is TelemetryRecord.__dict__['scaled_reading']

record.reading = 19
assert record.scaled_reading == 190
record.scale = 1
assert record.scaled_reading == 19
expect_error(ValueError, lambda: setattr(record, 'reading', 101), '[0, 100]')
expect_error(ValueError, lambda: setattr(record, 'scale', 2), 'scale')
assert (record.reading, record.scale, record.scaled_reading) == (19, 1, 19)
assert peer_record.scaled_reading == 8
print('Capstone functional checks passed.')

Expected ValueError: _reading_value must be an integer in [0, 100]
Expected ValueError: scale must be exactly 1 or 10
Capstone functional checks passed.


### Step 6 — verify reconstruction, independence, and non-registration

The registry is a *runtime observer*, not a persisted field. Newly deserialized records should not silently become members of the original registry. The record's logical state is serialized; its weak references and cache entries are reconstructed lazily.

In [153]:
restored = pickle.loads(pickle.dumps(record))
cloned = copy.deepcopy(record)
assert restored is not record and cloned is not record
assert (restored.reading, restored.scale) == (19, 1)
assert (cloned.reading, cloned.scale) == (19, 1)
assert restored.scaled_reading == 19 and cloned.scaled_reading == 19
assert registry.count() == 2

restored.reading = 77
cloned.scale = 10
assert restored.scaled_reading == 77
assert cloned.scaled_reading == 190
assert record.scaled_reading == 19
print('Capstone serialization checks passed.')

Capstone serialization checks passed.


### Step 7 — test disappearance in an isolated scope

The local registry survives after the owners are deleted; it should end up empty. Its snapshot method is intentionally dangerous to retain because a list of live instances is a normal **strong** reference container. Avoid keeping a snapshot around when checking lifetime.

In [154]:
def capstone_lifetime_case():
    live = LiveRegistry()
    a = TelemetryRecord(40, 10, live)
    b = TelemetryRecord(90, 1, live)
    descriptor = TelemetryRecord._reading_value
    derived = TelemetryRecord.scaled_reading
    baseline_values = len(descriptor._values)
    baseline_cached = len(derived._values)

    assert a.scaled_reading == 400
    assert b.scaled_reading == 90
    first_handle, second_handle = weakref.ref(a), weakref.ref(b)
    assert live.count() == 2
    del a, b
    gc.collect()
    assert first_handle() is None and second_handle() is None
    assert live.count() == 0
    assert len(descriptor._values) == baseline_values - 2
    assert len(derived._values) == baseline_cached


capstone_lifetime_case()
print('Problem 14 — all capstone checks passed.')

Problem 14 — all capstone checks passed.


### Step 8 — review the trade-offs

This implementation intentionally requires identity-hashable, weak-referenceable objects. Both `WeakKeyDictionary` maps strongly retain their numeric values but not their keys. The registry adds no strong owner references. A snapshot **does** temporarily retain owners, so discard it once used. Validation occurs before mutation and invalidation; pickling saves logical state rather than weakref internals.

There is one important integration detail: the descriptor's public `__set__` is technically accessible via `TelemetryRecord._reading_value.__set__(...)`, allowing deliberate bypass of the property. Python does not provide private access enforcement. For normal API use, direct `.reading = ...` always invalidates the cache; for a larger library, unify mutation under one carefully documented interface.

**Extensions for you to attempt:** (1) add a `reading` reset operation that invalidates the derived cache; (2) make a reusable dependency-aware cached descriptor; (3) design a transaction to update reading and scale atomically, validating both before changing either; (4) adapt the record to value-based equality, documenting why the weak storage choice must change.

---
## Final synthesis — choose storage based on actual constraints

| Need | Candidate | Non-negotiable caveat |
|---|---|---|
| Normal owner with `__dict__` | `__set_name__` plus instance-owned state | Slotted, dictionary-free objects cannot use it. |
| Hashable + weak-referenceable owners | `WeakKeyDictionary` | Equality/hash semantics matter; values can still retain owners. |
| Weakly retain expensive values | `WeakValueDictionary` | Cache entries may vanish without other strong owners. |
| Observe a live collection | `WeakSet` | Requires weak-referenceable, hashable members; set equality applies. |
| Weakly retain a bound instance method | `WeakMethod` | Retrieve and temporarily own the method while calling. |
| Delegate without owning the target | `weakref.proxy` | Operations raise `ReferenceError` after destruction. |
| Cleanup when an owner disappears | `weakref.finalize` | Callback and arguments must not strongly retain the owner. |
| Persist descriptor-managed state | Explicit `__getstate__`/`__setstate__` | Save logical data, not the weak-reference machinery. |
| Unhashable owner without an instance dictionary | Identity plus weakref callback, as in the supplied lecture | Requires weak-reference support and careful cleanup/ID-reuse defense. |

**Accuracy notes relative to the source:** An `id()`-keyed dictionary alone avoids strong owner keys but does **not** clean stale entries or prevent address-reuse bugs. Weakref callbacks cannot recover the already-dead referent. Weak-reference support is not the same as hashability. The original lecture's `ctypes`-based inspection of freed addresses is deliberately not reproduced because dereferencing deallocated memory is unsafe.

**What to practice next:** Re-run the notebook after replacing one weak mapping with a normal dictionary and predict which assertion fails. Then restore the weak mapping and intentionally put an owner reference inside a cached value; explain the resulting lifetime graph. Finally remove `__weakref__` from a fully slotted class and identify which operations immediately raise `TypeError`.

In [155]:
# Whole-notebook completion indicator: every prior cell must have executed successfully.
assert callable(weakref.ref)
assert all(name in globals() for name in (
    'NamedPositiveInteger', 'WeakEventBus', 'ResourceOwner',
    'WeakScore', 'WeakCachedTotal', 'TelemetryRecord',
))
print('SUCCESS: All 14 tutorial problems and their solution checks completed.')

SUCCESS: All 14 tutorial problems and their solution checks completed.
